# AI工学101 — 第25回

## 不均衡データと分類閾値：Recallを上げると何が起こる？

第24回では、**「Accuracyが高いモデル＝良いモデルとは限らない」**ことを学びました。

今日はそこから一歩進んで、

> **モデルが出した「確率」を、どの値でYes/Noに変換するのか？**

という問題を扱います。

ここはかなり重要。

実は `predict()` が返している

```text
0 / 1
```

は、モデルが直接「0か1か」を出しているわけではありません。

多くの場合、

```text
予測確率
   ↓
閾値で切る
   ↓
0 / 1
```

という処理が入っています。

そして、この**閾値を変えるだけでPrecisionとRecallのバランスを変えられる**。

今日は実際にコードを書いて、それを体感します。

---

# 🎯 今日のゴール

今日できるようになること：

* クラス不均衡が何か説明できる
* Accuracyが不均衡データで危険な理由を理解する
* `predict_proba()` の確率を使って分類できる
* 分類閾値（threshold）を自分で変更できる
* Precision / Recall のトレードオフを実験できる
* `class_weight="balanced"` の意味を理解する
* ROC曲線・Precision-Recall曲線の基本を理解する

---

# 📖 講義：約20〜25分

## 1. クラス不均衡とは？

例えば1000件のデータがあって、

```text
正常       950件
異常        50件
```

だったとします。

割合は、

```text
正常：95%
異常：5%
```

です。

これを**クラス不均衡（class imbalance）**と呼びます。

---

## 2. 全部「正常」と予測してみる

AIに、

```text
全部正常です
```

と言わせます。

すると、

```text
950 / 1000 = 95%
```

なのでAccuracyは、

**95%**

です。

でも、

```text
異常50件

↓

50件全部見逃し
```

です。

これはかなり危険。

だから、

> **不均衡データではAccuracyだけを見るのは危険**

となります。

---

# 🧠 3. 今日の重要概念：Threshold

例えばモデルが、

```python
predict_proba()
```

で、

```text
0.91
0.73
0.62
0.48
0.21
```

という確率を出したとします。

普通は、

```text
0.5以上 → 1
0.5未満 → 0
```

という感じで分類します。

つまり、

```text
threshold = 0.5
```

です。

でも、

```text
threshold = 0.3
```

にしたら？

```text
0.3以上 → 1
```

なので、もっと多くのデータを陽性と判定します。

---

# 🔥 4. Thresholdを下げると何が起きる？

例えば、

```text
threshold = 0.8
```

なら、

「かなり自信があるものだけ陽性」

になります。

すると、

```text
陽性と判定する数 ↓
```

なので、

```text
False Positive ↓
```

になりやすい。

一方で、

```text
本当は陽性なのに0.8未満
```

というケースを見逃しやすくなります。

つまり、

```text
Recall ↓
```

しやすい。

---

逆に、

```text
threshold = 0.2
```

なら、

「ちょっとでも怪しかったら陽性」

になります。

すると、

```text
本当の陽性を拾いやすい
```

ので、

```text
Recall ↑
```

しやすい。

ただし、

```text
健康なのに陽性
```

も増えやすいので、

```text
Precision ↓
```

しやすくなります。

---

# 🌳 今日の構造

```text
thresholdを高くする
        ↓
陽性判定が厳しくなる
        ↓
Precision ↑しやすい
Recall ↓しやすい


thresholdを低くする
        ↓
陽性判定が緩くなる
        ↓
Recall ↑しやすい
Precision ↓しやすい
```

これを

**Precision-Recall trade-off**

と呼びます。

---

# 💻 実習1：不均衡データを作る

今日は人工データを使います。

```python
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.95, 0.05],
    random_state=42
)
```

確認。

```python
import numpy as np

print(
    np.bincount(y)
)
```

例えば、

```text
[1900  100]
```

のようになります。

つまり、

```text
クラス0：95%
クラス1：5%
```

です。

かなり不均衡です。

---

# 💻 実習2：train/test分割

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

ここでも、

```python
stratify=y
```

を使っています。

train/testの両方で、

```text
クラス0：約95%

クラス1：約5%
```

という比率を保ちやすくするためです。

---

# 💻 実習3：Logistic Regression

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

学習。

```python
model.fit(
    X_train,
    y_train
)
```

---

# 💻 実習4：まず普通にpredict()

```python
pred = model.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        pred
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        pred
    )
)
```

まずこれを確認します。

---

# 💻 実習5：予測確率を見る

```python
prob = model.predict_proba(
    X_test
)[:, 1]
```

確認。

```python
print(
    prob[:20]
)
```

例えば、

```text
0.02
0.03
0.91
0.27
0.72
...
```

のような値になります。

これは、

> **クラス1であるとモデルが考えている確率**

です。

---

# 🧠 `predict()` の正体

実は、

```python
model.predict(X_test)
```

を自分で書き換えると、概念的にはこうできます。

```python
pred = (
    prob >= 0.5
).astype(int)
```

つまり、

```text
確率
 ↓
0.5で切る
 ↓
0 / 1
```

です。

これを実際に比較。

```python
pred_manual = (
    prob >= 0.5
).astype(int)
```

```python
print(
    np.array_equal(
        pred,
        pred_manual
    )
)
```

通常の二値分類では、同じ結果になるはずです。

---

# 💻 実習6：Thresholdを0.3にする

ここから面白くなります。

```python
threshold = 0.3

pred_03 = (
    prob >= threshold
).astype(int)
```

評価。

```python
print(
    "Precision:",
    precision_score(
        y_test,
        pred_03
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred_03
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        pred_03
    )
)
```

---

# 💻 実習7：Thresholdを0.7にする

```python
threshold = 0.7

pred_07 = (
    prob >= threshold
).astype(int)
```

```python
print(
    "Precision:",
    precision_score(
        y_test,
        pred_07
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred_07
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        pred_07
    )
)
```

---

# 🔥 実習8：Thresholdを全部試す

ここが今日のメイン実験。

```python
thresholds = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9
]
```

ループ。

```python
for threshold in thresholds:

    pred_threshold = (
        prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        pred_threshold,
        zero_division=0
    )

    print(
        "threshold =", threshold,
        "precision =", precision,
        "recall =", recall,
        "f1 =", f1
    )
```

---

# 👀 観察する

おそらく、

```text
threshold ↓

Recall ↑
```

という傾向が見えてきます。

一方、

```text
threshold ↑

Precision ↑
```

という傾向も見えるでしょう。

もちろんデータによって完全な単調増減になるとは限りません。

でも、

> **閾値を動かすことでPrecisionとRecallのバランスを変えられる**

ということが重要です。

---

# 📖 5. ROC曲線

ここから少し評価指標側に戻ります。

ROC曲線では、

```text
Threshold
```

をいろいろ変えながら、

```text
True Positive Rate
```

と

```text
False Positive Rate
```

の関係を見ます。

---

# 💻 実習9：ROC Curve

```python
from sklearn.metrics import (
    roc_curve,
    roc_auc_score
)
```

計算。

```python
fpr, tpr, thresholds = roc_curve(
    y_test,
    prob
)
```

AUC。

```python
auc = roc_auc_score(
    y_test,
    prob
)

print(
    "ROC-AUC:",
    auc
)
```

---

# 💻 実習10：ROC曲線を描く

```python
import matplotlib.pyplot as plt

plt.plot(
    fpr,
    tpr
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve"
)

plt.show()
```

ここで、

```text
横軸：False Positive Rate
縦軸：True Positive Rate
```

です。

---

# 📖 6. Precision-Recall Curve

不均衡データでは、

**Precision-Recall Curve**

も重要です。

読み込み。

```python
from sklearn.metrics import (
    precision_recall_curve
)
```

計算。

```python
precision, recall, thresholds = (
    precision_recall_curve(
        y_test,
        prob
    )
)
```

---

# 💻 実習11：PR曲線を描く

```python
plt.plot(
    recall,
    precision
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve"
)

plt.show()
```

ここでは、

```text
Recall
```

と

```text
Precision
```

のトレードオフを直接見ることができます。

---

# 🧠 ROCとPRの使い分け

ざっくり言うと、

### ROC

```text
全体的なランキング性能
```

を見るのに便利。

### Precision-Recall

```text
陽性クラスが少ない
```

ような不均衡問題では、特に重要になります。

今回のように、

```text
陽性5%
陰性95%
```

なら、

**PR曲線を確認する価値が高い**

ということです。

---

# 💻 実習12：class_weight="balanced"

もう一つ重要な方法があります。

モデル側に、

> 「クラス数が偏っているから、少数クラスをもっと重視して学習してね」

と伝える方法です。

```python
model_balanced = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )
    )
])
```

学習。

```python
model_balanced.fit(
    X_train,
    y_train
)
```

予測。

```python
pred_balanced = (
    model_balanced.predict(
        X_test
    )
)
```

評価。

```python
print(
    "Precision:",
    precision_score(
        y_test,
        pred_balanced
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred_balanced
    )
)
```

---

# 🧠 `class_weight="balanced"` とThresholdは違う

ここ重要です。

### `class_weight`

**学習そのもの**に影響します。

```text
モデルを学習するとき

↓

少数クラスを重視
```

---

### Threshold

**学習後の判定方法**を変えます。

```text
学習済みモデル

↓

予測確率

↓

threshold

↓

0 / 1
```

つまり、

```text
class_weight
    ↓
モデルをどう学習するか

threshold
    ↓
学習済みモデルをどう使うか
```

です。

この区別はかなり大事。

---

# 🧪 実習13：Threshold探索関数を作る

ここまでを関数にまとめます。

```python
def evaluate_thresholds(
    y_true,
    prob,
    thresholds
):

    for threshold in thresholds:

        pred = (
            prob >= threshold
        ).astype(int)

        precision = precision_score(
            y_true,
            pred,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            pred,
            zero_division=0
        )

        print(
            threshold,
            precision,
            recall,
            f1
        )
```

呼び出し。

```python
evaluate_thresholds(
    y_test,
    prob,
    np.arange(
        0.1,
        1.0,
        0.1
    )
)
```

---

# ✍️ 演習

## 問1

次のデータについて考えてください。

```text
正常：990
異常：10
```

「全部正常」と予測したとき、Accuracyはいくつになるでしょう？

そして、

> **なぜそれだけでは良いモデルと言えないのか？**

説明してください。

---

## 問2

`predict()` と `predict_proba()` の違いを説明してください。

---

## 問3

次のコードの意味を説明してください。

```python
pred = (
    prob >= 0.3
).astype(int)
```

---

## 問4

Thresholdを、

```text
0.2
0.5
0.8
```

に変えたとき、

PrecisionとRecallは一般にどのように変化しやすいでしょうか？

---

## 問5

`class_weight="balanced"` は、

**学習前後のどちらに作用する仕組みでしょうか？**

そしてThreshold変更との違いを説明してください。

---

# 👾 ボス戦：分類器を「運用」する

今日の本丸。

人工データについて、

```text
class 0：95%
class 1：5%
```

の状態でLogistic Regressionを学習します。

そして、

```text
threshold
precision
recall
f1
```

を一覧化してください。

さらに、

> **「異常を見逃すコストが非常に高いシステム」**

だと仮定します。

その場合、

```text
threshold = 0.5
```

をそのまま使うべきでしょうか？

それとも別のthresholdを検討するべきでしょうか？

**数値だけでなく、「なぜその閾値を選んだか」を説明すること。**

これが今日のボス戦です。

---

# 🌱 今日のまとめ

今日の一番重要なところは、

> **分類モデルの仕事は「0か1かを出すこと」だけではない。**

ということ。

実際には、

```text
入力
 ↓
モデル
 ↓
予測確率
 ↓
Threshold
 ↓
最終的な0 / 1
```

という構造になっています。

だから、

```text
モデルを変えなくても

Thresholdを変える
```

だけで、

```text
Recall
Precision
F1
```

のバランスを変えられます。

さらに、

```text
class_weight
```

を使えば、

**学習そのものを少数クラス重視にする**

こともできます。

---

# 🧭 現在地

第24回までで、

```text
モデルを作る
 ↓
評価する
```

から、

```text
何を失敗とする？
 ↓
評価指標を決める
 ↓
モデルを学習
 ↓
確率を出す
 ↓
Thresholdを決める
 ↓
運用上の性能を評価
```

というところまで来ました。

これはかなり重要な進歩です。

**AIモデルの性能は、アルゴリズムだけで決まるわけではない。**

```text
データ
+
特徴量
+
モデル
+
ハイパーパラメータ
+
評価指標
+
Threshold
```

まで含めて、ひとつのAIシステムとして設計します。

---

# 🔜 第26回

## 前処理の実戦：欠損値・カテゴリ変数・ColumnTransformer

次回から、いよいよ**現実の汚いデータ**に近づけます。

これまでは、

```text
全部数値
欠損なし
```

という、かなり綺麗なデータを使ってきました。

現実のデータはそんなに優しくありません。😂

そこで、

* 欠損値（Missing Values）
* 数値特徴量
* カテゴリ特徴量
* `SimpleImputer`
* `OneHotEncoder`
* `ColumnTransformer`
* それらをPipelineに統合する方法

を実装します。

ここで、

```text
生データ
 ↓
「汚い」
 ↓
前処理
 ↓
機械学習モデル
```

という**実際のデータ処理パイプライン**を作れるようにします。

scikit-learn編が、いよいよ「おもちゃデータ」から実務データへ寄っていくぞ。